# HackAlem AI — стартовый ноутбук

Ноутбук рассчитан на быстрый старт за ограниченное время хакатона. Выберите один кейс, подключите датасет и замените baseline на решение, которое покрывает обязательные требования кейса.

Перед сдачей: не храните API-ключи в ноутбуке или GitHub, добавьте README и проверьте сценарий от входных данных до результата.

## 0. Установка библиотек

Запустите один раз после установки Python. Для повторного запуска ячейку можно пропустить.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
from pathlib import Path
import os
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, mean_absolute_error, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

warnings.filterwarnings('ignore')
load_dotenv()
ROOT = Path.cwd()
DATA_DIR = ROOT / 'data'
OUTPUT_DIR = ROOT / 'outputs'
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
print('Python:', sys.version.split()[0])
print('Рабочая папка:', ROOT.resolve())

## 1. Быстрая проверка данных

Положите CSV/Parquet в папку `data/`. Код автоматически покажет таблицы и пропуски.

In [ ]:
files = sorted([*DATA_DIR.glob('*.csv'), *DATA_DIR.glob('*.parquet')])
if not files:
    print('Добавьте файл данных в папку data/')
else:
    for path in files:
        df_preview = pd.read_csv(path) if path.suffix == '.csv' else pd.read_parquet(path)
        print(f'\n{path.name}: {df_preview.shape}')
        display(df_preview.head())
        display(df_preview.isna().mean().sort_values(ascending=False).head(10).to_frame('доля пропусков'))

## 2. Baseline для табличной задачи

Укажите путь к данным и целевой столбец. Для классификации используйте `TASK = 'classification'`, для числового прогноза — `TASK = 'regression'`.

In [ ]:
DATA_PATH = DATA_DIR / 'train.csv'  # TODO: замените на имя файла
TARGET = 'target'                 # TODO: замените на целевой столбец
TASK = 'classification'            # 'classification' или 'regression'

if DATA_PATH.exists() and TARGET in pd.read_csv(DATA_PATH, nrows=1).columns:
    df = pd.read_csv(DATA_PATH)
    X = df.drop(columns=[TARGET])
    y = df[TARGET]
    numeric_features = X.select_dtypes(include=np.number).columns.tolist()
    categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()
    preprocessor = ColumnTransformer([
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features),
    ])
    estimator = RandomForestClassifier(n_estimators=250, random_state=42, n_jobs=-1) if TASK == 'classification' else RandomForestRegressor(n_estimators=250, random_state=42, n_jobs=-1)
    model = Pipeline([('preprocessor', preprocessor), ('model', estimator)])
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y if TASK == 'classification' else None)
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    if TASK == 'classification':
        print('Accuracy:', round(accuracy_score(y_test, predictions), 4))
        print(classification_report(y_test, predictions))
    else:
        print('MAE:', round(mean_absolute_error(y_test, predictions), 4))
else:
    print('Настройте DATA_PATH и TARGET, затем перезапустите ячейку.')

## 3. Безопасная интеграция OpenAI/NVIDIA

Ключи храните только в `.env` локально. Файл `.env` уже добавлен в `.gitignore`.

In [ ]:
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
NVIDIA_API_KEY = os.getenv('NVIDIA_API_KEY')
print('OPENAI_API_KEY:', 'задан' if OPENAI_API_KEY else 'не задан')
print('NVIDIA_API_KEY:', 'задан' if NVIDIA_API_KEY else 'не задан')

# Пример вызова OpenAI (раскомментируйте после добавления ключа):
# from openai import OpenAI
# client = OpenAI(api_key=OPENAI_API_KEY)
# response = client.responses.create(model='gpt-4.1-mini', input='Кратко опиши идею проекта.')
# print(response.output_text)

## 4. Чек-лист перед сдачей

- выбран один кейс и выполнены обязательные требования;
- основной сценарий проверен от входных данных до результата;
- README описывает назначение, функции, архитектуру, установку, проверку, данные, интеграции и ограничения;
- секреты удалены из ноутбука и репозитория;
- ноутбук запускается с чистого окружения;
- последняя версия отправлена в GitHub-репозиторий команды.